In [5]:
import json
import pandas as pd
import hashlib

# Ensure the ADOMD.NET path is in the system path prior to importing Pyadomd
from sys import path
adomd_path = r'C:\Program Files\Microsoft.NET\ADOMD.NET\160'
if adomd_path not in path:
    path.append(adomd_path)

# for x in path:
#     print(x, sep='\n')

from pyadomd import Pyadomd

json_path = "Sales & Returns Sample v201912.json"

with open(json_path, "r", encoding="utf-8-sig") as f:
    data = json.load(f)

# Flatten metrics into top-level columns
def flatten_event(event):
    flat = event.copy()
    metrics = flat.pop("metrics", {})
    flat.update(metrics)
    return flat

flat_events = [flatten_event(e) for e in data.get("events", [])]

df = pd.DataFrame(flat_events)

pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', 20)    
#print(df.head())

filtered_df = df[df['name'] == 'Execute DAX Query'].copy()

# filtered_df = df[
#     (df['name'] == 'Execute DAX Query') &
#     (df['id'] == 'baa3553b-c0fa-4f0e-87d6-98ea260fecba')
# ].copy()
filtered_df


# Connection string to your Power BI semantic model (update as needed)
connection_string = "Provider=MSOLAP;Data Source=localhost:61954;Initial Catalog=31499e00-1fda-40a9-b59e-a2227c83ab3d"


def row_hash(row):
    # Concatenate all values as string and hash
    row_str = '|'.join(str(val) for val in row.values)
    return hashlib.sha256(row_str.encode('utf-8')).hexdigest()


# Loop through the DataFrame and execute each DAX query
with Pyadomd(connection_string) as conn:
    #Loop over each query from the dataframe
    for idx, row in filtered_df.iterrows():
        query_id = row['id']
        query_text = row['QueryText']
        print(f"QueryText:\n{query_text[:50]}\n")
        #print(f"Executing Query ID: {query_id}")

        try:
            with conn.cursor().execute(query_text) as cur:
                result_set_index = 1
                query_hashes = []  # List to collect all query hashes
                has_next = True
                while has_next:
                    columns = [col[0] for col in cur.description]
                    result_rows = cur.fetchall()
                    print(f"Result set {result_set_index} has {len(result_rows)} rows")

                    if result_rows:
                        # Create a DataFrame for this query's results
                        result_df = pd.DataFrame(result_rows, columns=columns)

                        # Add a hash column for each row
                        if not result_df.empty:
                            result_df['row_hash'] = result_df.apply(row_hash, axis=1)

                            # Sort by row_hash
                            result_df = result_df.sort_values('row_hash').reset_index(drop=True)
                            # Concatenate all row_hash values and hash them
                            row_hashes = '|'.join(result_df['row_hash'].tolist())
                            combined_row_hash = hashlib.sha256(row_hashes.encode('utf-8')).hexdigest()

                            query_hashes.append(combined_row_hash)

                            # Store in filtered_df (for the first result set only, or adjust as needed)
                            # if result_set_index == 0:
                            #     filtered_df.loc[idx, 'combined_row_hash'] = combined_row_hash
                            print(f"query: {query_id} result set {result_set_index} rows: {len(result_df)} combined hash: {combined_row_hash}")
                        else:
                            print(f"query: {query_id} result set {result_set_index} is empty")

                    result_set_index += 1
                    has_next = cur.nextresult()

                if query_hashes:
                    combined = '|'.join(query_hashes)
                    combined_query_hash = hashlib.sha256(combined.encode('utf-8')).hexdigest()
                    print(f"Final combined hash: {combined_query_hash}")

                    # Store in filtered_df (for the first result set only, or adjust as needed)
                    filtered_df.loc[idx, 'combined_query_hash'] = combined_query_hash

        except Exception as e:
            print(f"Error executing query {query_id}: {e}\n")



QueryText:
DEFINE VAR __DS0FilterTable = 
	TREATAS({"June"},

Result set 1 has 1 rows
query: 82bd947a-c0cb-48f5-87aa-2db65515b25e result set 1 rows: 1 combined hash: dfaa90b9f1d0497f2836e7a73b6e332125ba378ff023382491970708da76b0d5
Final combined hash: bf3b72873b105c202bd18b2025d0b6becbcd63c5ab45b04704efd50dd91ed9f4
QueryText:
DEFINE
	VAR __DS0FilterTable = 
		TREATAS({"June

Result set 1 has 1 rows
query: baa3553b-c0fa-4f0e-87d6-98ea260fecba result set 1 rows: 1 combined hash: dfaa90b9f1d0497f2836e7a73b6e332125ba378ff023382491970708da76b0d5
Result set 2 has 33 rows
query: baa3553b-c0fa-4f0e-87d6-98ea260fecba result set 2 rows: 33 combined hash: f05c4829ddfeca46de5f15ca46f2c09f8efad35d27b681ce5934562a0522ea7a
Final combined hash: f337c81e916a742acf92fa3a99a94d5b2fa616fe79311f81e3e411c2ebce641b
QueryText:
DEFINE
	VAR __DS0FilterTable = 
		TREATAS({"Sold

Result set 1 has 1 rows
query: 09fa85a8-b263-484c-a590-1fb11a89288a result set 1 rows: 1 combined hash: a587cdfbde2b7d9e4235afdeadd82d6